# Preprocessing Wavelet Filter + Intensity Normalization

## 1. Instalasi & Import Library

In [ ]:
!pip install PyWavelets scikit-image opencv-python-headless matplotlib numpy pandas scipy -q

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pywt
import warnings
warnings.filterwarnings('ignore')

from skimage import img_as_float, img_as_ubyte
from scipy.signal import welch

## 2. Mount Google Drive & Konfigurasi Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
INPUT_BASE  = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_roi_preprocessed_test'
OUTPUT_BASE = '/content/drive/MyDrive/Tugas Akhir/FINAL_WN_test'

DATASET_SPLITS = {
    'train' : {'no_tumor': f'{INPUT_BASE}/train/no_tumor',
               'tumor'   : f'{INPUT_BASE}/train/tumor'},
    'val'   : {'no_tumor': f'{INPUT_BASE}/val/no_tumor',
               'tumor'   : f'{INPUT_BASE}/val/tumor'},
    'test'  : {'no_tumor': f'{INPUT_BASE}/test/no_tumor',
               'tumor'   : f'{INPUT_BASE}/test/tumor'},
}

CLASSES      = ['no_tumor', 'tumor']
SAMPLE_COUNT = 10   # jumlah sampel per kelas per split untuk evaluasi metrik

for split in DATASET_SPLITS:
    for cls in CLASSES:
        os.makedirs(os.path.join(OUTPUT_BASE, split, cls), exist_ok=True)

## 3. Fungsi Helper

In [ ]:
def load_rgb(path: str) -> np.ndarray:
    """Load gambar sebagai RGB float [0, 1], shape (H, W, 3)."""
    img_bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if img_bgr is None:
        raise FileNotFoundError(f'Gambar tidak ditemukan: {path}')
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return img_as_float(img_rgb)


def get_image_paths(folder: str, n: int = None) -> list:
    """Ambil daftar path gambar dari sebuah folder."""
    if not os.path.isdir(folder):
        return []
    exts  = ('.png', '.jpg', '.jpeg', '.bmp')
    paths = sorted([os.path.join(folder, f)
                    for f in os.listdir(folder)
                    if f.lower().endswith(exts)])
    return paths[:n] if n else paths


total = 0
for split, cls_map in DATASET_SPLITS.items():
    for cls, folder in cls_map.items():
        n = len(get_image_paths(folder))
        total += n
        if n == 0:
            print(f'[WARN] Folder kosong/tidak ada: {split}/{cls}')

print(f'Total dataset: {total} gambar')

## 4. Implementasi Wavelet Filter (Noise Reduction)

In [ ]:
def apply_wavelet(
    img_float : np.ndarray,
    wavelet   : str   = 'db1',
    level     : int   = 1,
    threshold : float = 0.05,
) -> np.ndarray:
    """Wavelet soft-thresholding; RGB diproses per-channel independen."""
    if img_float.ndim == 2:
        return _apply_wavelet_2d(img_float, wavelet, level, threshold)

    channels = []
    for c in range(img_float.shape[2]):
        ch_denoised = _apply_wavelet_2d(img_float[:, :, c], wavelet, level, threshold)
        channels.append(ch_denoised)
    return np.stack(channels, axis=2)


def _apply_wavelet_2d(
    img_2d    : np.ndarray,
    wavelet   : str   = 'db1',
    level     : int   = 1,
    threshold : float = 0.05,
) -> np.ndarray:
    """Wavelet thresholding untuk array 2D single-channel."""
    coeffs = pywt.wavedec2(img_2d, wavelet, level=level)
    coeffs_thresh = [coeffs[0]] + [
        tuple(pywt.threshold(c, threshold, mode='soft') for c in detail)
        for detail in coeffs[1:]
    ]
    reconstructed = pywt.waverec2(coeffs_thresh, wavelet)
    h, w = img_2d.shape
    return np.clip(reconstructed[:h, :w], 0, 1)

## 5. Implementasi Intensity Normalization (Image Enhancement)

In [ ]:
def apply_intensity_normalization(
    img_float  : np.ndarray,
    norm_stats : dict = None,
) -> np.ndarray:
    """Z-score per-channel: (x-mean)/std, dari statistik train set (hindari leakage).
    norm_stats=None -> fallback min-max per-gambar (khusus evaluasi metrik)."""
    if norm_stats is None:
        img_out = np.empty_like(img_float)
        if img_float.ndim == 3:
            for c in range(img_float.shape[2]):
                ch = img_float[:, :, c]
                lo, hi = ch.min(), ch.max()
                img_out[:, :, c] = (ch - lo) / (hi - lo) if (hi - lo) > 1e-8 else ch.copy()
        else:
            lo, hi = img_float.min(), img_float.max()
            img_out = (img_float - lo) / (hi - lo) if (hi - lo) > 1e-8 else img_float.copy()
        return img_out

    mean = np.array(norm_stats['mean'], dtype=np.float64)
    std  = np.array(norm_stats['std'],  dtype=np.float64)
    std  = np.maximum(std, 1e-7)

    img_out = (img_float.astype(np.float64) - mean) / std
    return img_out.astype(np.float32)


def compute_and_save_norm_stats(
    dataset_splits : dict,
    output_base    : str,
    train_splits   : list = None,
) -> dict:
    """Hitung mean/std per-channel dari train set (streaming, hemat memori) -> simpan JSON."""
    import json as _json
    from pathlib import Path
    from tqdm.notebook import tqdm

    if train_splits is None:
        train_splits = [k for k in dataset_splits if 'train' in k]

    channel_sum    = np.zeros(3, dtype=np.float64)
    channel_sum_sq = np.zeros(3, dtype=np.float64)
    n_pixels = 0

    all_files = []
    for split_name in train_splits:
        cls_map = dataset_splits.get(split_name, {})
        for cls, folder in cls_map.items():
            all_files.extend(get_image_paths(folder))

    if not all_files:
        raise RuntimeError('Tidak ada file gambar di train split. Cek path DATASET_SPLITS.')

    for img_path in tqdm(all_files, desc='Hitung mean/std', unit='img'):
        img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            continue
        arr = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float64) / 255.0
        h, w, _ = arr.shape
        n_pixels       += h * w
        channel_sum    += arr.sum(axis=(0, 1))
        channel_sum_sq += (arr ** 2).sum(axis=(0, 1))

    mean = channel_sum / n_pixels
    std  = np.sqrt(np.maximum(channel_sum_sq / n_pixels - mean ** 2, 0))
    std  = np.maximum(std, 1e-7)

    stats = {
        'mean'           : mean.tolist(),
        'std'            : std.tolist(),
        'channels'       : ['R', 'G', 'B'],
        'scale'          : '0-1 (pixel / 255.0)',
        'n_train_images' : len(all_files),
        'n_pixels_total' : int(n_pixels),
        'train_splits'   : train_splits,
    }

    stats_path = str(Path(output_base) / 'norm_stats_wavelet.json')
    with open(stats_path, 'w') as f:
        _json.dump(stats, f, indent=2)

    print(f'Mean R,G,B = {[round(float(v), 4) for v in mean]}')
    print(f'Std  R,G,B = {[round(float(v), 4) for v in std]}')
    print(f'Disimpan: {stats_path}')
    return stats

## 6. Pipeline Final: Wavelet -> Intensity Normalization

In [ ]:
def full_pipeline(path: str, norm_stats: dict = None) -> np.ndarray:
    """Pipeline: load RGB -> Wavelet Filter -> Intensity Normalization (z-score)."""
    img = load_rgb(path)
    img = apply_wavelet(img)
    img = apply_intensity_normalization(img, norm_stats)
    return img

## 12. Batch Processing - Simpan Semua Gambar ke Disk

## 12a. Simpan Contoh Hasil - Before & After per Tahap

In [ ]:
import shutil

SAMPLE_PREPROC_DIR = os.path.join(OUTPUT_BASE, 'sample_preprocessing')
SAMPLE_N           = 5   # jumlah sampel before/after yang disimpan

def save_sample_png(img_float: np.ndarray, out_path: str) -> None:
    """Simpan array RGB float [0,1] ke PNG (clip dulu ke [0,1])."""
    clipped = np.clip(img_float, 0, 1)
    img_bgr = cv2.cvtColor(img_as_ubyte(clipped), cv2.COLOR_RGB2BGR)
    cv2.imwrite(out_path, img_bgr)

_sample_paths_candidate = (
    get_image_paths(DATASET_SPLITS['train']['tumor'],    n=SAMPLE_N)
    or get_image_paths(DATASET_SPLITS['train']['no_tumor'], n=SAMPLE_N)
)
sample_paths_12a = _sample_paths_candidate[:SAMPLE_N]

if not sample_paths_12a:
    print('[WARN] Tidak ada gambar di train/tumor maupun train/no_tumor.')
else:
    # Tahap 1: Wavelet Filter
    dir_wav_before = os.path.join(SAMPLE_PREPROC_DIR, 'wavelet', 'before')
    dir_wav_after  = os.path.join(SAMPLE_PREPROC_DIR, 'wavelet', 'after')
    os.makedirs(dir_wav_before, exist_ok=True)
    os.makedirs(dir_wav_after,  exist_ok=True)

    for idx, path in enumerate(sample_paths_12a):
        original = load_rgb(path)
        wavelet  = apply_wavelet(original)
        save_sample_png(original, os.path.join(dir_wav_before, f'before_{idx}.png'))
        save_sample_png(wavelet,  os.path.join(dir_wav_after,  f'after_{idx}.png'))

    # Tahap 2: Intensity Normalization (before=wavelet, after=hasil normalisasi)
    dir_norm_before = os.path.join(SAMPLE_PREPROC_DIR, 'intensity_norm', 'before')
    dir_norm_after  = os.path.join(SAMPLE_PREPROC_DIR, 'intensity_norm', 'after')
    os.makedirs(dir_norm_before, exist_ok=True)
    os.makedirs(dir_norm_after,  exist_ok=True)

    # Fallback JSON/min-max jika Sel 12b belum dijalankan (norm_stats_wavelet belum ada)
    _norm_stats_preview = None
    try:
        _norm_stats_preview = norm_stats_wavelet
    except NameError:
        import json as _json2
        _stats_path = os.path.join(OUTPUT_BASE, 'norm_stats_wavelet.json')
        if os.path.isfile(_stats_path):
            with open(_stats_path) as _f:
                _norm_stats_preview = _json2.load(_f)
        else:
            print('[WARN] norm_stats_wavelet belum tersedia -> fallback min-max per-gambar')

    for idx, path in enumerate(sample_paths_12a):
        original = load_rgb(path)
        wavelet  = apply_wavelet(original)
        normed   = apply_intensity_normalization(wavelet, _norm_stats_preview)
        save_sample_png(wavelet, os.path.join(dir_norm_before, f'before_{idx}.png'))
        save_sample_png(normed,  os.path.join(dir_norm_after,  f'after_{idx}.png'))

    print(f'Sample tersimpan di: {SAMPLE_PREPROC_DIR}')

## 12c. Jalankan Pipeline & Simpan Seluruh Dataset

In [ ]:
total_saved = 0
total_error = 0

for split, cls_map in DATASET_SPLITS.items():
    for cls, folder in cls_map.items():
        paths   = get_image_paths(folder)
        out_dir = os.path.join(OUTPUT_BASE, split, cls)
        os.makedirs(out_dir, exist_ok=True)

        folder_error = 0

        for path in paths:
            try:
                result = full_pipeline(path, norm_stats=norm_stats_wavelet)

                fname    = os.path.splitext(os.path.basename(path))[0] + '.png'
                out_path = os.path.join(out_dir, fname)

                # Setelah z-score, nilai bisa di luar [0,1] -> clip eksplisit sebelum imwrite
                result_clipped = np.clip(result, 0, 1)
                result_bgr     = cv2.cvtColor(img_as_ubyte(result_clipped), cv2.COLOR_RGB2BGR)
                cv2.imwrite(out_path, result_bgr)

                total_saved += 1
            except Exception as e:
                print(f'[WARN] {path}: {e}')
                folder_error += 1
                total_error  += 1

        if folder_error > 0:
            print(f'[WARN] {split}/{cls}: {folder_error} gambar gagal diproses')

print(f'Selesai! {total_saved} gambar disimpan, {total_error} error.')

## 12d. Copy Masks - Tanpa Preprocessing

In [ ]:
import shutil

total_masks_copied  = 0
total_masks_skipped = 0

for split, cls_map in DATASET_SPLITS.items():
    for cls, folder in cls_map.items():
        masks_src = os.path.join(folder, 'masks')
        if not os.path.isdir(masks_src):
            continue

        masks_dst = os.path.join(OUTPUT_BASE, split, cls, 'masks')
        os.makedirs(masks_dst, exist_ok=True)

        mask_files = []
        for root_dir, dirs, files in os.walk(masks_src):
            for fname in files:
                mask_files.append(os.path.join(root_dir, fname))

        copied_count = 0
        for src_file in mask_files:
            rel_path = os.path.relpath(src_file, masks_src)
            dst_file = os.path.join(masks_dst, rel_path)
            os.makedirs(os.path.dirname(dst_file), exist_ok=True)
            shutil.copy2(src_file, dst_file)
            copied_count       += 1
            total_masks_copied += 1

        if copied_count == 0:
            total_masks_skipped += 1
            print(f'[WARN] Folder masks ada tapi kosong: {masks_src}')

if total_masks_copied == 0 and total_masks_skipped == 0:
    print('Tidak ditemukan subfolder masks/ di manapun dalam INPUT_BASE.')
else:
    print(f'Selesai! Total {total_masks_copied} file mask disalin ke OUTPUT_BASE.')